# Tome 11, Chapter 1: Gaussian processes and the birth of Brownian motion (Python companion)

Exercises first, solutions at the end. Three ways to simulate Brownian motion (random walk, Lévy's midpoints, the Karhunen–Loève series) and how their errors compare, Kolmogorov's criterion seen on a computer, fractional Brownian motion by exact Cholesky sampling, and a computer science application: **estimating the Hurst exponent** of a time series, the statistic behind Hurst's study of the Nile and behind modern tests for long memory in network traffic. Requires `numpy` (and `matplotlib` for the optional pictures).

In [ ]:
import math
import numpy as np
rng = np.random.default_rng(1101)

## Exercises

**P1.1 (★) Three Brownian motions.** Build a Brownian path on $[0,1]$ at the $2^{10}+1$ dyadic points in three ways: (a) cumulative sums of independent $N(0,2^{-10})$ steps; (b) Lévy's midpoint construction (Tome 3, Theorem 11.4): start from $B_0=0$, $B_1\sim N(0,1)$, and at level $n$ set each new midpoint to the average of its neighbours plus an independent $N(0,2^{-n-2})$, the variance of a Brownian bridge at the middle of an interval of length $2^{-n}$; (c) the Karhunen–Loève series of Theorem 1.10 with $N$ terms. For each method, estimate $\operatorname{Var}B_{1/2}$, $\operatorname{Cov}(B_{1/4},B_{3/4})$ and $\mathbb E\max_tB_t$ from $5000$ paths, and compare with $\frac12$, $\frac14$ and $\sqrt{2/\pi}$ (Tome 3, Theorem 11.12). Which method is biased for the maximum, and why?

**P1.2 (★) How fast does the series converge?** Theorem 1.10 says the Karhunen–Loève partial sums $V^{(N)}$ converge uniformly. Estimate $\mathbb E\|V^{(2N)}-V^{(N)}\|_\infty$ for $N=8,16,\dots,512$ and fit a power law. Compare with the bound $c\sqrt{\log N/N}$ of the proof. And compare $\mathbb E\int_0^1(V-V^{(N)})^2$ with the exact value $\sum_{n>N}((n-\frac12)\pi)^{-2}\approx\frac1{\pi^2N}$.

**P1.3 (★★) Kolmogorov's criterion on a computer.** For a Brownian path at the dyadic points of level $n=14$, compute $\Delta_m=\max_k|B_{(k+1)2^{-m}}-B_{k2^{-m}}|$ for $m=1,\dots,14$, and plot $\log_2\Delta_m$ against $m$. What slope do you expect from Corollary 1.6? Compare $\Delta_m$ with $\sqrt{2\cdot2^{-m}\log 2^m}$ (Lévy's modulus, Chapter 2).

**P1.4 (★★) Fractional Brownian motion.** Sample fractional Brownian motion at $n=512$ points for $H\in\{0.25,0.5,0.75\}$ by the Cholesky factorization of the covariance of Theorem 1.12. (a) Check the empirical correlation of consecutive increments against $2^{2H-1}-1$. (b) Check self-similarity: the law of $X_{1/2}$ should be that of $2^{-H}X_1$. (c) Compute $Q_n=\sum(X_{j/n}-X_{(j-1)/n})^2$ and compare with $n^{1-2H}$ (Exercise 1.8).

**P1.5 (★★★) CS application: estimating the Hurst exponent.** The *aggregated variance* method: for a series of increments $y_1,\dots,y_n$, average it over blocks of length $m$, compute the variance $v(m)$ of the block means, and fit $v(m)\propto m^{2H-2}$. (a) Implement it and test it on fractional Gaussian noise (the increments of P1.4) with known $H$. (b) Test it on i.i.d. noise and on a random walk (which is *not* stationary; what goes wrong?). (c) Hurst's rescaled range $R/S$: implement it and compare its bias with the aggregated variance method for $H=0.75$ and $n=4096$. Long memory in traffic traces (Leland, Taqqu, Willinger and Wilson, 1994) was detected exactly this way.

## Solutions

### P1.1
**How to find it.** All three give exactly Gaussian values with the right covariance *at the grid points* when done right (a and b), so the covariances must agree; the maximum over $1025$ points is smaller than the maximum over the continuum, by an amount of order $\sqrt{2^{-10}}$. The truncated series (c) is smoother than Brownian motion, so its maximum is also slightly too small, and its variance at $t=\frac12$ misses the tail $\sum_{n>N}$.

In [ ]:
n = 10; M = 2**n; m = 5000
t = np.arange(M + 1) / M

def walk(m):
    steps = rng.standard_normal((m, M)) / math.sqrt(M)
    return np.concatenate([np.zeros((m, 1)), np.cumsum(steps, axis=1)], axis=1)

def levy(m):
    B = np.zeros((m, M + 1)); B[:, M] = rng.standard_normal(m)
    h = M
    level = 0
    while h > 1:
        mid = np.arange(h // 2, M, h)
        B[:, mid] = 0.5 * (B[:, mid - h // 2] + B[:, mid + h // 2]) \
            + rng.standard_normal((m, len(mid))) * math.sqrt(2.0**(-level - 2))
        h //= 2; level += 1
    return B

def kl(m, N=400):
    k = np.arange(1, N + 1)
    F = math.sqrt(2) * np.sin(np.outer(t, (k - 0.5) * np.pi)) / ((k - 0.5) * np.pi)
    return rng.standard_normal((m, N)) @ F.T

for name, gen in [("walk", walk), ("Levy", levy), ("KL 400 terms", kl)]:
    B = gen(m)
    print(f"{name:13s} Var B_1/2 = {B[:, M//2].var():.3f}  Cov(B_1/4,B_3/4) = {np.mean(B[:, M//4]*B[:, 3*M//4]):.3f}"
          f"  E max = {B.max(axis=1).mean():.3f}")
print("exact:        0.5, 0.25, sqrt(2/pi) =", round(math.sqrt(2/math.pi), 3))

In Lévy's construction the midpoint at level $n$ (counting from $0$) has conditional variance $2^{-n-2}$: the variance of a Brownian bridge at the middle of an interval of length $h=2^{-n}$ is $h/4$. The walk and Lévy's construction are *exact* at the grid points; all three underestimate $\mathbb E\max$ by about $0.58\sqrt{2^{-10}}\approx0.02$, the discretization error of the maximum, and the truncated series a little more.

### P1.2
**How to find it.** Sample once with $1024$ coefficients and take the partial sums from the same coefficients, as in Figure 1.1.

In [ ]:
tt = np.linspace(0, 1, 2001)
K = 1024
k = np.arange(1, K + 1)
F = math.sqrt(2) * np.sin(np.outer(tt, (k - 0.5) * np.pi)) / ((k - 0.5) * np.pi)
xi = rng.standard_normal((300, K))
Ns, gaps = [], []
N = 8
while 2 * N <= K:
    d = xi[:, N:2*N] @ F[:, N:2*N].T
    Ns.append(N); gaps.append(np.abs(d).max(axis=1).mean())
    N *= 2
slope = np.polyfit(np.log(Ns), np.log(gaps), 1)[0]
for N, g in zip(Ns, gaps):
    print(f"N={N:4d}  E||V^(2N)-V^(N)|| = {g:.4f}   sqrt(log N / N) = {math.sqrt(math.log(N)/N):.4f}")
print("fitted exponent:", round(slope, 3), "(sqrt(log N / N) has local slope -1/2 + 1/(2 log N), about -0.4 here)")
print("L2 tail for N=100:", sum(1/((n-0.5)*math.pi)**2 for n in range(101, 10**6)), "vs 1/(pi^2 N) =", 1/(math.pi**2*100))

### P1.3
**How to find it.** At scale $2^{-m}$ the $2^m$ increments are independent $N(0,2^{-m})$, and the largest of $2^m$ such Gaussians is about $\sqrt{2^{-m}\cdot2\log 2^m}$. So $\log_2\Delta_m\approx-\frac m2+\frac12\log_2(2m\log2)$: slope $-\frac12$, slightly flattened by the logarithm. This is why the Hölder exponent is $\frac12^-$ and not $\frac12$.

In [ ]:
n = 14; M = 2**n
B = np.concatenate([[0], np.cumsum(rng.standard_normal(M) / math.sqrt(M))])
for m in range(1, n + 1):
    step = 2**(n - m)
    D = np.abs(np.diff(B[::step])).max()
    print(f"m={m:2d}  Delta_m = {D:.4f}   Levy modulus sqrt(2 h log(1/h)) = {math.sqrt(2 * 2**-m * m * math.log(2)):.4f}")

### P1.4
**How to find it.** The covariance matrix $C_{jk}=\frac12(t_j^{2H}+t_k^{2H}-|t_j-t_k|^{2H})$ is positive definite on distinct positive times; $X=Lz$ with $C=LL^{\mathsf T}$ and $z$ standard Gaussian has covariance $C$ (Tome 3, Theorem 6.15).

In [ ]:
def fbm(H, n, m):
    ts = np.arange(1, n + 1) / n
    S, T = np.meshgrid(ts, ts)
    C = 0.5 * (S**(2*H) + T**(2*H) - np.abs(S - T)**(2*H))
    L = np.linalg.cholesky(C)
    X = rng.standard_normal((m, n)) @ L.T
    return np.concatenate([np.zeros((m, 1)), X], axis=1)

n = 512
for H in (0.25, 0.5, 0.75):
    X = fbm(H, n, 400)
    dX = np.diff(X, axis=1)
    r = np.mean(dX[:, :-1] * dX[:, 1:]) / np.mean(dX**2)
    Q = (dX**2).sum(axis=1).mean()
    print(f"H={H}: lag-1 corr {r:+.3f} (exact {2**(2*H-1)-1:+.3f});"
          f" Var X_1/2 / Var X_1 = {X[:, n//2].var()/X[:, n].var():.3f} (exact {2**(-2*H):.3f});"
          f" Q_n = {Q:.2f} (exact {n**(1-2*H):.2f})")

### P1.5
**How to find it.** For stationary increments with $\operatorname{Var}(y_1+\dots+y_m)=m^{2H}$, the block mean has variance $m^{2H-2}$: a straight line of slope $2H-2$ on a log–log plot. A random walk is not stationary: its "increments" are themselves the walk, whose block means have variance growing with the position, so the method returns nonsense ($H$ near $1$). The rescaled range is known to be biased upwards for short series.

In [ ]:
def agg_var_H(y, ms=(4, 8, 16, 32, 64, 128)):
    v = []
    for m in ms:
        k = len(y) // m
        v.append(y[:k*m].reshape(k, m).mean(axis=1).var())
    slope = np.polyfit(np.log(ms), np.log(v), 1)[0]
    return 1 + slope / 2

def rs_H(y, ms=(16, 32, 64, 128, 256, 512)):
    out = []
    for m in ms:
        k = len(y) // m
        blocks = y[:k*m].reshape(k, m)
        Z = np.cumsum(blocks - blocks.mean(axis=1, keepdims=True), axis=1)
        R = Z.max(axis=1) - Z.min(axis=1)
        S = blocks.std(axis=1)
        out.append(np.mean(R / S))
    return np.polyfit(np.log(ms), np.log(out), 1)[0]

for H in (0.25, 0.5, 0.75):
    y = np.diff(fbm(H, 2048, 1)[0])
    print(f"fGn H={H}: aggregated variance {agg_var_H(y):.2f}, R/S {rs_H(y):.2f}")
y = rng.standard_normal(2048)
print("iid noise:", round(agg_var_H(y), 2), "  random walk treated as increments:", round(agg_var_H(np.cumsum(y)), 2))

Both estimators find the right side of $\frac12$, but neither is exact at this length: the aggregated variance method is biased downwards for $H=0.75$ (the block means of a long-memory series estimate their own mean badly), and the rescaled range is biased towards $\frac12$ for small $H$. Real studies use longer series and wavelet or maximum likelihood estimators. Applied to a random walk it gives $H\approx1$: always difference a non-stationary series before estimating its memory.